In [18]:
import os
import sys
import re
import json
import difflib
import ijson
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# ── 项目根目录解析 ──
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from src.utils.cwe import CWEUtils
from src.utils.cve import CVEUtils
from src.utils.formatter import auto_format_c_code_batch

# ── 超参数 ──
MIN_CODE_LEN     = 35       # 最短代码字符数（过滤过短的无意义代码）
MAX_CODE_LEN     = 65535    # 最长代码字符数（Milvus VARCHAR 上限）
MERGE_FUNCS_AS_CVE = True  # 是否按 (CVE ID, commit_id) 合并同源多函数
# OUTPUT_PATH      = PROJECT_ROOT / "data/raw/vul/BigVul.jsonl"
OUTPUT_PATH      = PROJECT_ROOT / "data/raw/vul/BigVul_Merged.jsonl"


# ── 流式加载 + 即时过滤原始数据 ──
# BigVul.json 顶层为字典（键为行号字符串），原始大小 ~11GB。
# 若先全部 append 再 DataFrame，会把 ~50GB 数据驻留内存导致 WSL2 OOM。
# 这里在 ijson 迭代过程中就完成最关键的过滤，只保留后续真正需要的字段，
# 使得 DataFrame 内存占用从 ~50GB 压到 ~几百MB。
dataset_path = Path("dataset/BigVul.json")

# 后续 cell 实际会用到的原始字段（重命名前）
KEEP_FIELDS = (
    "func_before",  # → vul_code
    "func_after",   # → fixed_code
    "CWE ID",       # → cwe_id
    "CVE ID",       # → cve_id
    "lang",         # → language
    "Summary",      # → cve_desc_raw
    "commit_id",    # MERGE_FUNCS_AS_CVE=True 时使用
    "vul",          # 给 cell 1 步骤 1 兜底（虽然下面已过滤）
)

def _quick_valid(val):
    """轻量级的字符串非空判断，避免 cell 1 才发现需要丢弃。"""
    if not isinstance(val, str):
        return False
    s = val.strip()
    return len(s) > 0 and s.lower() != "none"

records = []
total = 0
with open(dataset_path, "rb") as f:
    for _, record in tqdm(ijson.kvitems(f, ""), desc="流式过滤 BigVul"):
        total += 1

        # 流式过滤①：仅保留漏洞样本 vul == 1（数据集中大部分是 vul == 0 的负样本）
        if str(record.get("vul", "")).strip() != "1":
            continue

        # 流式过滤②：仅保留 C / C++
        if record.get("lang") not in ("C", "C++"):
            continue

        # 流式过滤③：vul_code / fixed_code 必须非空且不相等，
        #            同时长度落在 [MIN_CODE_LEN, MAX_CODE_LEN] 区间内
        vul_code = record.get("func_before")
        fixed_code = record.get("func_after")
        if not _quick_valid(vul_code) or not _quick_valid(fixed_code):
            continue
        if vul_code == fixed_code:
            continue
        if not (MIN_CODE_LEN <= len(vul_code) <= MAX_CODE_LEN):
            continue
        if not (MIN_CODE_LEN <= len(fixed_code) <= MAX_CODE_LEN):
            continue

        # 流式过滤④：CWE / CVE 标识必须非空
        if not _quick_valid(record.get("CWE ID")) or not _quick_valid(record.get("CVE ID")):
            continue

        # 字段白名单：只保留后续真正会用到的列，丢弃其余十几列大字段
        records.append({k: record.get(k) for k in KEEP_FIELDS})

df_raw = pd.DataFrame(records)
del records  # 立即释放原始列表，归还内存
print(f"原始样本数:       {total}")
print(f"流式过滤后样本数:  {len(df_raw)}")
df_raw.head()

Project root: <PROJECT_ROOT>


流式过滤 BigVul: 188636it [13:34, 231.54it/s] 

原始样本数:       188636
流式过滤后样本数:  8570


,func_before,func_after,CWE ID,CVE ID,lang,Summary,commit_id,vul
0,static int samldb_check_user_account_control_a...,static int samldb_check_user_account_control_a...,CWE-264,CVE-2015-8467,C,The samldb_check_user_account_control_acl func...,b000da128b5fb519d2d3f2e7fd20e4a25b7dae7d,1
1,PHPAPI void php_pcre_match_impl(pcre_cache_ent...,PHPAPI void php_pcre_match_impl(pcre_cache_ent...,CWE-119,CVE-2015-8382,C,The match function in pcre_exec.c in PCRE befo...,c351b47ce85a3a147cfa801fa9f0149ab4160834,1
2,static int scan(Scanner *s)\n{\n\tuchar *curso...,static int scan(Scanner *s)\n{\n\tuchar *curso...,CWE-119,CVE-2013-6712,C,The scan function in ext/date/lib/parse_iso_in...,12fe4e90be7bfa2a763197079f68f5568a14e071,1
3,long ssl_get_algorithm2(SSL *s)\n {\n ...,long ssl_get_algorithm2(SSL *s)\n {\n ...,CWE-310,CVE-2013-6449,C,The ssl_get_algorithm2 function in ssl/s3_lib....,ca989269a2876bae79393bd54c3e72d49975fc75,1
4,static time_t asn1_time_to_time_t(ASN1_UTCTIME...,static time_t asn1_time_to_time_t(ASN1_UTCTIME...,CWE-119,CVE-2013-6420,C,The asn1_time_to_time_t function in ext/openss...,c1224573c773b6845e83505f717fbf820fc18415,1


In [21]:
df = df_raw.copy()

def _is_valid_str(val):
    """判断值是否为非空、非 None 的字符串。"""
    if val is None:
        return False
    if not isinstance(val, str):
        return False
    s = val.strip()
    return len(s) > 0 and s.lower() != 'none'

# ── 步骤 1：只保留漏洞样本（vul == 1） ──
df['vul'] = df['vul'].astype(str).str.strip()
df = df[df['vul'] == '1']
print(f"步骤1 (vul==1):           {len(df):>7} 条")

# ── 步骤 2：只保留 C 或 C++ 语言 ──
df = df[df['lang'].isin(['C', 'C++'])]
print(f"步骤2 (C/C++ 语言):        {len(df):>7} 条")

# ── 步骤 3：字段重命名 ──
df = df.rename(columns={
    'func_before': 'vul_code',
    'func_after':  'fixed_code',
    'CWE ID':      'cwe_id',
    'CVE ID':      'cve_id',
    'lang':        'language',
    'Summary':     'cve_desc_raw',
})

# ── 步骤 4：vul_code 和 fixed_code 非空 ──
df = df[
    df['vul_code'].apply(_is_valid_str)
    & df['fixed_code'].apply(_is_valid_str)
]
print(f"步骤4 (代码非空):          {len(df):>7} 条")

# ── 步骤 5：vul_code != fixed_code（漏洞修复真实有效） ──
df = df[df['vul_code'] != df['fixed_code']]
print(f"步骤5 (vul!=fixed):        {len(df):>7} 条")

# ── 步骤 6：cwe_id 和 cve_id 非空 ──
df = df[
    df['cwe_id'].apply(_is_valid_str)
    & df['cve_id'].apply(_is_valid_str)
]
print(f"步骤6 (CWE/CVE非空):       {len(df):>7} 条")

# ── 步骤 7：代码长度过滤 ──
vul_len   = df['vul_code'].str.len()
fixed_len = df['fixed_code'].str.len()
df = df[
    vul_len.between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')
    & fixed_len.between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')
]
print(f"步骤7 (代码长度合法):      {len(df):>7} 条")

df = df.reset_index(drop=True)
print(f"\n清洗后总样本数: {len(df)}")
df[['cwe_id', 'cve_id', 'language', 'vul_code', 'fixed_code']].head()

步骤1 (vul==1):              8570 条
步骤2 (C/C++ 语言):           8570 条
步骤4 (代码非空):             8570 条
步骤5 (vul!=fixed):           8570 条
步骤6 (CWE/CVE非空):          8570 条
步骤7 (代码长度合法):         8570 条

清洗后总样本数: 8570


,cwe_id,cve_id,language,vul_code,fixed_code
0,CWE-264,CVE-2015-8467,C,static int samldb_check_user_account_control_a...,static int samldb_check_user_account_control_a...
1,CWE-119,CVE-2015-8382,C,PHPAPI void php_pcre_match_impl(pcre_cache_ent...,PHPAPI void php_pcre_match_impl(pcre_cache_ent...
2,CWE-119,CVE-2013-6712,C,static int scan(Scanner *s)\n{\n\tuchar *curso...,static int scan(Scanner *s)\n{\n\tuchar *curso...
3,CWE-310,CVE-2013-6449,C,long ssl_get_algorithm2(SSL *s)\n {\n ...,long ssl_get_algorithm2(SSL *s)\n {\n ...
4,CWE-119,CVE-2013-6420,C,static time_t asn1_time_to_time_t(ASN1_UTCTIME...,static time_t asn1_time_to_time_t(ASN1_UTCTIME...


In [22]:
SEP = "\n\n/* ---- next function ---- */\n\n"

if MERGE_FUNCS_AS_CVE:
    print("启用同源 CVE 合并 (MERGE_FUNCS_AS_CVE=True)...")

    # 注意：直接使用 groupby(...).apply(...) 在新版 pandas 中存在分组键与返回值列名冲突
    # 的问题（cve_id/commit_id 会进入 MultiIndex，与 Series 同名键冲突后被静默丢弃，
    # 导致 cve_id 列在结果中消失）。这里改为手动遍历分组并显式构造列表，行为更可控。
    merged_rows = []
    for (cve_id, commit_id), grp in df.groupby(['cve_id', 'commit_id'], sort=False):
        first = grp.iloc[0].to_dict()
        first['cve_id']     = cve_id
        first['commit_id']  = commit_id
        first['vul_code']   = SEP.join(grp['vul_code'].tolist())
        first['fixed_code'] = SEP.join(grp['fixed_code'].tolist())
        merged_rows.append(first)

    df = pd.DataFrame(merged_rows).reset_index(drop=True)

    # 合并后再次过滤超长代码
    df = df[
        df['vul_code'].str.len().between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')
        & df['fixed_code'].str.len().between(MIN_CODE_LEN, MAX_CODE_LEN, inclusive='both')
    ].reset_index(drop=True)

    print(f"合并后样本数: {len(df)}")
else:
    print(f"跳过同源 CVE 合并 (MERGE_FUNCS_AS_CVE=False)，当前样本数: {len(df)}")

df[['cve_id', 'cwe_id', 'language', 'vul_code', 'fixed_code']].head()

启用同源 CVE 合并 (MERGE_FUNCS_AS_CVE=True)...
合并后样本数: 3142


,cve_id,cwe_id,language,vul_code,fixed_code
0,CVE-2015-8467,CWE-264,C,static int samldb_check_user_account_control_a...,static int samldb_check_user_account_control_a...
1,CVE-2015-8382,CWE-119,C,PHPAPI void php_pcre_match_impl(pcre_cache_ent...,PHPAPI void php_pcre_match_impl(pcre_cache_ent...
2,CVE-2013-6712,CWE-119,C,static int scan(Scanner *s)\n{\n\tuchar *curso...,static int scan(Scanner *s)\n{\n\tuchar *curso...
3,CVE-2013-6449,CWE-310,C,long ssl_get_algorithm2(SSL *s)\n {\n ...,long ssl_get_algorithm2(SSL *s)\n {\n ...
4,CVE-2013-6420,CWE-119,C,static time_t asn1_time_to_time_t(ASN1_UTCTIME...,static time_t asn1_time_to_time_t(ASN1_UTCTIME...


In [23]:
from src.utils.formatter import auto_format_c_code_batch, ClangFormatStyle

print("格式化 vul_code ...")
vul_results = auto_format_c_code_batch(
    df['vul_code'].tolist(),
    style=ClangFormatStyle.LLVM,
    on_error="keep",
    show_progress=True,
)
df['vul_code'] = [r[0] for r in vul_results]

print("格式化 fixed_code ...")
fixed_results = auto_format_c_code_batch(
    df['fixed_code'].tolist(),
    style=ClangFormatStyle.LLVM,
    on_error="keep",
    show_progress=True,
)
df['fixed_code'] = [r[0] for r in fixed_results]

# 统计格式化置信度情况
low_conf_vul   = sum(1 for r in vul_results   if r[1].confidence < 0.6)
low_conf_fixed = sum(1 for r in fixed_results if r[1].confidence < 0.6)
print(f"\nvul_code   低置信度样本数 (<0.6): {low_conf_vul}")
print(f"fixed_code 低置信度样本数 (<0.6): {low_conf_fixed}")
print(f"格式化完成，总样本数: {len(df)}")

# ── 步骤5-bis：格式化后再次过滤 vul_code == fixed_code ──
# clang-format 会将仅存在空白符/缩进差异的代码对统一为相同字符串，
# 因此必须在格式化后重新执行此过滤，才能保证 diff 非空。
before_dedup = len(df)
df = df[df['vul_code'] != df['fixed_code']].reset_index(drop=True)
removed = before_dedup - len(df)
print(f"\n步骤5-bis (格式化后 vul!=fixed): 移除 {removed} 条，剩余 {len(df)} 条")

格式化 vul_code ...


Formatting: 100%|██████████| 3142/3142 [00:22<00:00, 141.78snippet/s]


格式化 fixed_code ...


Formatting: 100%|██████████| 3142/3142 [00:20<00:00, 156.56snippet/s]



vul_code   低置信度样本数 (<0.6): 123
fixed_code 低置信度样本数 (<0.6): 116
格式化完成，总样本数: 3142

步骤5-bis (格式化后 vul!=fixed): 移除 1 条，剩余 3141 条


In [24]:
cwe_utils = CWEUtils(PROJECT_ROOT / "data/raw/cwe/cwec_v4.19.1.xml")

def get_cwe_desc(cwe_field):
    """将 cwe_id 字段转换为 CWEUtils prompt 文本。

    支持字符串 / 列表 / numpy 数组格式；找不到时返回 None。
    """
    if cwe_field is None:
        return None
    cwe_id = None
    if isinstance(cwe_field, np.ndarray) and len(cwe_field) > 0:
        cwe_id = str(cwe_field[0])
    elif isinstance(cwe_field, list) and len(cwe_field) > 0:
        cwe_id = str(cwe_field[0])
    elif isinstance(cwe_field, str):
        m = re.search(r"CWE-\d+", cwe_field, re.IGNORECASE)
        cwe_id = m.group() if m else cwe_field.strip().strip("[]'\"").split(",")[0].strip().strip("'\"")
    if not cwe_id or not cwe_id.strip():
        return None
    prompt = cwe_utils.get_prompt(cwe_id.strip())
    if "not found in the CWE dataset" in prompt:
        return None
    return prompt

df['cwe_desc'] = df['cwe_id'].apply(get_cwe_desc)

filled = df['cwe_desc'].notna().sum()
print(f"总样本数:              {len(df)}")
print(f"cwe_desc 已填充:       {filled}")
print(f"cwe_desc 为空:         {len(df) - filled}")
print(f"cwe_desc 填充率:       {filled / len(df) * 100:.2f}%")
df[['cwe_id', 'cwe_desc']].head()

总样本数:              3141
cwe_desc 已填充:       3141
cwe_desc 为空:         0
cwe_desc 填充率:       100.00%


,cwe_id,cwe_desc
0,CWE-264,"- CWE ID: CWE-264\n- Name: Permissions, Privil..."
1,CWE-119,- CWE ID: CWE-119\n- Name: Improper Restrictio...
2,CWE-119,- CWE ID: CWE-119\n- Name: Improper Restrictio...
3,CWE-310,- CWE ID: CWE-310\n- Name: Cryptographic Issue...
4,CWE-119,- CWE ID: CWE-119\n- Name: Improper Restrictio...


In [25]:
cve_utils = CVEUtils(PROJECT_ROOT / "data/raw/cwe/CVE_v2026.03.18.json")

def _normalize_cve_desc_raw(val):
    """将 Summary 字段归一化为字符串，过滤空值。"""
    if val is None:
        return None
    if isinstance(val, str):
        s = val.strip()
        return s if s and s.lower() != 'none' else None
    return None

def get_cve_desc(row):
    """CVE 描述获取策略：优先使用原始 Summary 字段，不足时用 CVEUtils 补充。"""
    # 1. 优先使用 BigVul 原始 Summary 字段
    raw_desc = _normalize_cve_desc_raw(row.get('cve_desc_raw'))
    if raw_desc:
        return raw_desc

    # 2. 回退：通过 CVEUtils 按 cve_id 查询
    cve_id = row.get('cve_id')
    if not cve_id or not str(cve_id).strip():
        return None
    prompt = cve_utils.get_prompt(str(cve_id).strip())
    if "was not found in the CVE dataset" in prompt:
        return None
    return prompt

df['cve_desc'] = df.apply(get_cve_desc, axis=1)

filled   = df['cve_desc'].notna().sum()
from_raw = df['cve_desc_raw'].apply(_normalize_cve_desc_raw).notna().sum()
print(f"总样本数:                  {len(df)}")
print(f"cve_desc 已填充:           {filled}")
print(f"  └─ 来自 Summary 字段:    {from_raw}")
print(f"  └─ 来自 CVEUtils 补充:   {filled - from_raw}")
print(f"cve_desc 为空:             {len(df) - filled}")
print(f"cve_desc 填充率:           {filled / len(df) * 100:.2f}%")
df[['cve_id', 'cve_desc']].head()

总样本数:                  3141
cve_desc 已填充:           3141
  └─ 来自 Summary 字段:    3141
  └─ 来自 CVEUtils 补充:   0
cve_desc 为空:             0
cve_desc 填充率:           100.00%


,cve_id,cve_desc
0,CVE-2015-8467,The samldb_check_user_account_control_acl func...
1,CVE-2015-8382,The match function in pcre_exec.c in PCRE befo...
2,CVE-2013-6712,The scan function in ext/date/lib/parse_iso_in...
3,CVE-2013-6449,The ssl_get_algorithm2 function in ssl/s3_lib....
4,CVE-2013-6420,The asn1_time_to_time_t function in ext/openss...


In [26]:
def compute_diff(vul_code: str, fixed_code: str) -> str:
    """使用 difflib.unified_diff 计算函数级代码变更差异。"""
    a = vul_code.splitlines(keepends=True)
    b = fixed_code.splitlines(keepends=True)
    diff_lines = list(difflib.unified_diff(a, b, fromfile="vul_code", tofile="fixed_code"))
    return "".join(diff_lines) if diff_lines else ""

df['diff'] = df.apply(lambda r: compute_diff(r['vul_code'], r['fixed_code']), axis=1)

empty_diff = (df['diff'] == '').sum()
print(f"总样本数:         {len(df)}")
print(f"diff 非空:        {len(df) - empty_diff}")
print(f"diff 为空:        {empty_diff}  （vul_code 与 fixed_code 完全一致，理论上已在步骤5过滤）")
df[['vul_code', 'fixed_code', 'diff']].head(1)

总样本数:         3141
diff 非空:        3141
diff 为空:        0  （vul_code 与 fixed_code 完全一致，理论上已在步骤5过滤）


,vul_code,fixed_code,diff
0,static int samldb_check_user_account_control_a...,static int samldb_check_user_account_control_a...,"--- vul_code\n+++ fixed_code\n@@ -9,12 +9,15 @..."


In [27]:
df = df.reset_index(drop=True)
df['id']     = df.index        # 0-based 自增整数主键
df['source'] = 'BigVul'

print(f"id 范围:   0 ~ {df['id'].max()}")
print(f"source:    {df['source'].unique().tolist()}")
print(f"总样本数:  {len(df)}")
df[['id', 'source', 'cwe_id', 'cve_id', 'language']].head()

id 范围:   0 ~ 3140
source:    ['BigVul']
总样本数:  3141


,id,source,cwe_id,cve_id,language
0,0,BigVul,CWE-264,CVE-2015-8467,C
1,1,BigVul,CWE-119,CVE-2015-8382,C
2,2,BigVul,CWE-119,CVE-2013-6712,C
3,3,BigVul,CWE-310,CVE-2013-6449,C
4,4,BigVul,CWE-119,CVE-2013-6420,C


In [28]:
FINAL_COLS = [
    'id', 'source', 'language',
    'cwe_id', 'cve_id',
    'cwe_desc', 'cve_desc',
    'vul_code', 'fixed_code',
    'diff',
]

df_out = df[FINAL_COLS].copy()

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_out.to_json(
    OUTPUT_PATH,
    orient='records',
    lines=True,
    force_ascii=False,
)

print(f"导出完成: {OUTPUT_PATH}")
print(f"总样本数: {len(df_out)}")
print(f"\n字段统计：")
for col in FINAL_COLS:
    null_count = df_out[col].isna().sum()
    fill_rate  = (1 - null_count / len(df_out)) * 100
    print(f"  {col:<14}: 填充率 {fill_rate:6.2f}%  (空值 {null_count} 条)")
df_out.head()

导出完成: <PROJECT_ROOT>/data/raw/vul/BigVul_Merged.jsonl
总样本数: 3141

字段统计：
  id            : 填充率 100.00%  (空值 0 条)
  source        : 填充率 100.00%  (空值 0 条)
  language      : 填充率 100.00%  (空值 0 条)
  cwe_id        : 填充率 100.00%  (空值 0 条)
  cve_id        : 填充率 100.00%  (空值 0 条)
  cwe_desc      : 填充率 100.00%  (空值 0 条)
  cve_desc      : 填充率 100.00%  (空值 0 条)
  vul_code      : 填充率 100.00%  (空值 0 条)
  fixed_code    : 填充率 100.00%  (空值 0 条)
  diff          : 填充率 100.00%  (空值 0 条)


,id,source,language,cwe_id,cve_id,cwe_desc,cve_desc,vul_code,fixed_code,diff
0,0,BigVul,C,CWE-264,CVE-2015-8467,"- CWE ID: CWE-264\n- Name: Permissions, Privil...",The samldb_check_user_account_control_acl func...,static int samldb_check_user_account_control_a...,static int samldb_check_user_account_control_a...,"--- vul_code\n+++ fixed_code\n@@ -9,12 +9,15 @..."
1,1,BigVul,C,CWE-119,CVE-2015-8382,- CWE ID: CWE-119\n- Name: Improper Restrictio...,The match function in pcre_exec.c in PCRE befo...,PHPAPI void php_pcre_match_impl(pcre_cache_ent...,PHPAPI void php_pcre_match_impl(pcre_cache_ent...,"--- vul_code\n+++ fixed_code\n@@ -87,6 +87,7 @..."
2,2,BigVul,C,CWE-119,CVE-2013-6712,- CWE ID: CWE-119\n- Name: Improper Restrictio...,The scan function in ext/date/lib/parse_iso_in...,static int scan(Scanner *s) {\n uchar *cursor...,static int scan(Scanner *s) {\n uchar *cursor...,"--- vul_code\n+++ fixed_code\n@@ -147,7 +147,7..."
3,3,BigVul,C,CWE-310,CVE-2013-6449,- CWE ID: CWE-310\n- Name: Cryptographic Issue...,The ssl_get_algorithm2 function in ssl/s3_lib....,long ssl_get_algorithm2(SSL *s) {\n long alg2...,long ssl_get_algorithm2(SSL *s) {\n long alg2...,"--- vul_code\n+++ fixed_code\n@@ -1,6 +1,6 @@\..."
4,4,BigVul,C,CWE-119,CVE-2013-6420,- CWE ID: CWE-119\n- Name: Improper Restrictio...,The asn1_time_to_time_t function in ext/openss...,static time_t asn1_time_to_time_t(ASN1_UTCTIME...,static time_t asn1_time_to_time_t(ASN1_UTCTIME...,"--- vul_code\n+++ fixed_code\n@@ -13,20 +13,30..."
